# LoRA fine-tuning

Before starting to run cells top to bottom change runtime type to T4 GPU: Runtime > Change runtime type > pick a **T4 GPU** > Save.

If it says "too many active sessions," terminate background runtimes.

## Step 1: Check that you have a GPU

In [ ]:
!nvidia-smi

Thu Nov  5 22:08:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Install the libraries (takes about 1 minute)

In [ ]:
!pip install -q -U peft accelerate transformers

## Step 3: Upload your data files
Run the cell, click **Choose files**, and select all your `.jsonl` files at once
(the 5 open-model "fingerprinting" files, plus the 2 Gemini "training" files if you want the bonus run).

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving llama3.18bfingerprinting.jsonl to llama3.18bfingerprinting.jsonl
Saving llama3.23bfingerprinting.jsonl.json to llama3.23bfingerprinting.jsonl.json
Saving qwen3.59bfingerprinting.jsonl to qwen3.59bfingerprinting.jsonl
Saving qwen3.8bfingerprinting.jsonl.json to qwen3.8bfingerprinting.jsonl.json
Saving gemma34bfingerprinting.jsonl to gemma34bfingerprinting.jsonl
['llama3.18bfingerprinting.jsonl', 'llama3.23bfingerprinting.jsonl.json', 'qwen3.59bfingerprinting.jsonl', 'qwen3.8bfingerprinting.jsonl.json', 'gemma34bfingerprinting.jsonl']


## Step 4: Choose which run you are doing and prepare the data
- `"open_models"`: 5 classes (gemma3-4b, llama3.1-8b, llama3.2-3b, qwen3-8b, qwen3.5-9b). **Do this one first.**
- `"gemini"`: 2 classes (gemini-3.1-pro vs gemini-3.8-flash). Bonus run.

**Never mix the two groups in one run.** They used different questions, so the model could "cheat" by recognizing the question instead of the writing style.

In [ ]:
import json, glob, random, collections, re
import torch

RUN = "open_models"

if RUN == "open_models":
    FILE_PATTERN = "*fingerprinting*"
    MAX_TRAIN_PER_CLASS = 200
else:
    FILE_PATTERN = "*training*"
    MAX_TRAIN_PER_CLASS = 400

MAX_TOKENS = 512      # Increased from 384 to capture more stylistic text
SEED = 42

# 1) load every record
rows = []
for f in sorted(glob.glob(FILE_PATTERN)) :
    with open(f) as fh:
        for line in fh:
            if line.strip():
                rows.append(json.loads(line))
print("records loaded:", len(rows))

# 2) keep only records that have an actual answer
def has_answer(r):
    return (r.get("response_text") or "").strip() != "" and not r.get("error")
usable = [r for r in rows if has_answer(r)]
print("usable records:", len(usable), "| dropped:", len(rows) - len(usable))

# 3) split by QUESTION, not by record
prompt_group = {r["prompt_id"]: r["prompt_group"] for r in rows}
rng = random.Random(SEED)
test_prompts = set()
for g in sorted(set(prompt_group.values())):
    ids = sorted(i for i, x in prompt_group.items() if x == g)
    rng.shuffle(ids)
    test_prompts |= set(ids[: max(1, round(len(ids) * 0.2))])

train_all = [r for r in usable if r["prompt_id"] not in test_prompts]
test = [r for r in usable if r["prompt_id"] in test_prompts]

# 4) cap each class in the training set
rng.shuffle(train_all)
count = collections.Counter()
train = []
for r in train_all:
    if count[r["model_key"]] < MAX_TRAIN_PER_CLASS:
        count[r["model_key"]] += 1
        train.append(r)

labels = sorted(set(r["model_key"] for r in usable))
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

# Compute class weights to handle imbalance in Qwen 3.5 9B
train_counts = collections.Counter(r["model_key"] for r in train)
total_train = len(train)
class_weights_list = [total_train / (len(labels) * train_counts[l]) for l in labels]
class_weights = torch.tensor(class_weights_list, dtype=torch.float).to("cuda" if torch.cuda.is_available() else "cpu")

print("\nclasses:", labels)
print("train examples:", len(train), dict(train_counts))
print("test examples:", len(test), dict(collections.Counter(r["model_key"] for r in test)))
print("computed class weights:", {labels[i]: round(class_weights_list[i], 2) for i in range(len(labels))})

def family(name):
    return re.match(r"[a-z]+", name).group(0)

records loaded: 3840
usable records: 3087 | dropped: 753

classes: ['gemma3-4b', 'llama3.1-8b', 'llama3.2-3b', 'qwen3-8b', 'qwen3.5-9b']
train examples: 898 {'gemma3-4b': 200, 'llama3.2-3b': 200, 'llama3.1-8b': 200, 'qwen3-8b': 200, 'qwen3.5-9b': 98}
test examples: 697 {'gemma3-4b': 168, 'llama3.1-8b': 168, 'llama3.2-3b': 168, 'qwen3.5-9b': 39, 'qwen3-8b': 154}
computed class weights: {'gemma3-4b': 0.9, 'llama3.1-8b': 0.9, 'llama3.2-3b': 0.9, 'qwen3-8b': 0.9, 'qwen3.5-9b': 1.83}


## Step 5: A simple baseline to beat
This is a normal (non-LoRA) method that counts words. If LoRA can't beat this, that is still a valid result to report, and a good talking point for the mentor.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform([r["response_text"] for r in train])
clf = LogisticRegression(max_iter=2000).fit(Xtr, [r["model_key"] for r in train])
base_pred = clf.predict(vec.transform([r["response_text"] for r in test]))
base_true = [r["model_key"] for r in test]
print("Baseline accuracy (exact model):", round(accuracy_score(base_true, base_pred), 3))
print("Baseline accuracy (family only):", round(accuracy_score([family(x) for x in base_true], [family(x) for x in base_pred]), 3))
print("Random guessing would be about", round(1 / len(labels), 2))

Baseline accuracy (exact model): 0.631
Baseline accuracy (family only): 0.819
Random guessing would be about 0.2


## Step 6: Load the small open model and attach LoRA
`Qwen/Qwen2.5-1.5B` is a small open-weight model (about 1.5 billion numbers). The LoRA "sticky notes" are the only part that gets trained.
**If you get a "CUDA out of memory" error later**, change the name below to `Qwen/Qwen2.5-0.5B` and re-run from this cell.

In [ ]:
!pip uninstall -y -q torchao # added this cuz the LoRA library doesnt work with the torchao library. so restart session under runtime and rerun steps 4-6

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-1.5B" # Upgraded to 1.5B for higher pattern capacity

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id
)
model = model.float()
model.config.pad_token_id = tok.pad_token_id

lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,                 # Doubled rank for higher expressiveness
    lora_alpha=64,        # Doubled alpha scaling
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 36,937,216 || all params: 1,580,659,200 || trainable%: 2.3368


## Step 7: Turn the text into numbers the model can read

In [ ]:
from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding

class AnswerDataset(Dataset):
    def __init__(self, records):
        self.items = []
        for r in records:
            enc = tok(r["response_text"].strip(), truncation=True, max_length=MAX_TOKENS)
            enc["labels"] = label2id[r["model_key"]]
            self.items.append(enc)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

train_ds = AnswerDataset(train)
test_ds = AnswerDataset(test)
collator = DataCollatorWithPadding(tokenizer=tok)
print(len(train_ds), "train /", len(test_ds), "test")

898 train / 697 test


## Step 8: Train
This is the actual fine-tuning. Expect roughly 15 to 40 minutes on a free T4.
The `loss` number printed as it goes should trend **down** (it's how wrong the model is).

In [ ]:
# to update transformers to the latest version
!pip install -U transformers

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="lora_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=8,          # Doubled epochs for deeper convergence
    learning_rate=1e-4,          # Lower learning rate for stable adaptation on 1.5B
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collator
)
trainer.train()

Step,Training Loss
10,31.574725
20,18.832741
30,13.821710
40,9.792247
50,8.499841
60,5.012999
70,3.164754
80,2.437260
90,3.269989
100,3.764270


TrainOutput(global_step=456, training_loss=2.6008402526117207, metrics={'train_runtime': 2087.7402, 'train_samples_per_second': 3.441, 'train_steps_per_second': 0.218, 'total_flos': 2.4572741510946816e+16, 'train_loss': 2.6008402526117207, 'epoch': 8.0})

## Step 9: Test on the questions the model has never seen

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

out = trainer.predict(test_ds)
logits = out.predictions[0] if isinstance(out.predictions, tuple) else out.predictions
pred_ids = np.argmax(logits, axis=-1)
true_ids = [label2id[r["model_key"]] for r in test]

pred = [id2label[i] for i in pred_ids]
true = [id2label[i] for i in true_ids]

print("LoRA accuracy (exact model):", round(accuracy_score(true, pred), 3))
print("LoRA accuracy (family only):", round(accuracy_score([family(x) for x in true], [family(x) for x in pred]), 3))
print("Baseline was:", round(accuracy_score(base_true, base_pred), 3), "| random guessing:", round(1 / len(labels), 2))
print()
print(classification_report(true, pred, labels=labels, zero_division=0))
print("Confusion matrix (rows = real model, columns = what LoRA guessed):")
print("order:", labels)
print(confusion_matrix(true, pred, labels=labels))

LoRA accuracy (exact model): 0.676
LoRA accuracy (family only): 0.894
Baseline was: 0.631 | random guessing: 0.2

              precision    recall  f1-score   support

   gemma3-4b       0.95      0.88      0.91       168
 llama3.1-8b       0.46      0.46      0.46       168
 llama3.2-3b       0.49      0.43      0.46       168
    qwen3-8b       0.81      0.92      0.86       154
  qwen3.5-9b       0.64      0.82      0.72        39

    accuracy                           0.68       697
   macro avg       0.67      0.70      0.68       697
weighted avg       0.67      0.68      0.67       697

Confusion matrix (rows = real model, columns = what LoRA guessed):
order: ['gemma3-4b', 'llama3.1-8b', 'llama3.2-3b', 'qwen3-8b', 'qwen3.5-9b']
[[147   6   4   5   6]
 [  3  78  69  12   6]
 [  3  77  72  15   1]
 [  2   3   2 142   5]
 [  0   6   0   1  32]]


## Step 10: Try it on any text, and save your work

In [ ]:
def which_model(text):
    enc = tok(text, truncation=True, max_length=MAX_TOKENS, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits.float(), dim=-1)[0]
    ranked = sorted(zip(labels, probs.tolist()), key=lambda x: -x[1])
    return [(l, round(p, 3)) for l, p in ranked]

model.eval()
print(which_model(test[0]["response_text"]), "<- real answer:", test[0]["model_key"])

# save the LoRA "sticky notes" (small file) and download it
model.save_pretrained("my_lora_adapter")
!zip -r -q my_lora_adapter.zip my_lora_adapter
from google.colab import files
files.download("my_lora_adapter.zip")

[('gemma3-4b', 1.0), ('llama3.1-8b', 0.0), ('qwen3.5-9b', 0.0), ('qwen3-8b', 0.0), ('llama3.2-3b', 0.0)] <- real answer: gemma3-4b


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>